# **Maximizing Annual Memberships: Unveiling User Patterns at Cyclistic**

> This is a mock scenario from Google's data analytics certification course.

In 2016, Cyclistic launched a successful bike-share offering. Since then, the program has grown to a fleet of 5,824 bicycles that are geotracked and locked into a network of 692 stations across Chicago. The bikes can be unlocked from one station and returned to any other station in the system anytime.

Until now, Cyclistic’s marketing strategy relied on building general awareness and appealing to broad consumer segments. One approach that helped make these things possible was the flexibility of its pricing plans: single-ride passes, full-day passes, and annual memberships. 

* Customers who purchase single-ride or full-day passes are referred to as **casual** riders. 

* Customers who purchase annual memberships are Cyclistic **members**.

Cyclistic’s finance analysts have concluded that annual members are much more profitable than casual riders. Although the pricing flexibility helps Cyclistic attract more customers, Moreno believes that maximizing the number of annual members will be key to future growth. 

Rather than creating a marketing campaign that targets all-new customers, Moreno believes there is a very good chance to convert casual riders into members. She notes that casual riders are already aware of the Cyclistic program and have chosen Cyclistic for their mobility needs.

Moreno has set a clear goal: **Design marketing strategies aimed at converting casual riders into annual members.** 
In order to do that, however, the marketing analyst team needs to better understand how annual members and casual riders differ, why casual riders would buy a membership, and how digital media could affect their marketing tactics. Moreno and her team are interested in analyzing the Cyclistic historical bike trip data to identify trends.

# **Install and load libraries**

In [ ]:
install.packages("tidyverse")
install.packages("ggplot2")
install.packages("lubridate")
install.packages("dplyr")
install.packages("plotly")

In [ ]:
library(tidyverse)
library(ggplot2)
library(lubridate)
library(dplyr)
library(plotly)

# **Bring in the data!**

In [ ]:
Jan <- read_csv("/kaggle/input/divvy-trip-data-2022/202201-divvy-tripdata.csv")
Feb <- read_csv("/kaggle/input/divvy-trip-data-2022/202202-divvy-tripdata.csv")
Mar <- read_csv("/kaggle/input/divvy-trip-data-2022/202203-divvy-tripdata.csv")
Apr <- read_csv("/kaggle/input/divvy-trip-data-2022/202204-divvy-tripdata.csv")
May <- read_csv("/kaggle/input/divvy-trip-data-2022/202205-divvy-tripdata.csv")
Jun <- read_csv("/kaggle/input/divvy-trip-data-2022/202206-divvy-tripdata.csv")
Jul <- read_csv("/kaggle/input/divvy-trip-data-2022/202207-divvy-tripdata.csv")
Aug <- read_csv("/kaggle/input/divvy-trip-data-2022/202208-divvy-tripdata.csv")
Sep <- read_csv("/kaggle/input/divvy-trip-data-2022/202209-divvy-tripdata.csv")
Oct <- read_csv("/kaggle/input/divvy-trip-data-2022/202210-divvy-tripdata.csv")
Nov <- read_csv("/kaggle/input/divvy-trip-data-2022/202211-divvy-tripdata.csv")
Dec <- read_csv("/kaggle/input/divvy-trip-data-2022/202212-divvy-tripdata.csv")

Compare column names each of the files.

I'll need these to be consistent for combining the data later.

In [ ]:
colnames(Jan)
colnames(Feb)
colnames(Mar)
colnames(Apr)
colnames(May)
colnames(Jun)
colnames(Jul)
colnames(Aug)
colnames(Sep)
colnames(Oct)
colnames(Nov)
colnames(Dec)

Inspect the data frames and look for inconsistencies in structure.

In [ ]:
str(Jan)
str(Feb)
str(Mar)
str(Apr)
str(May)
str(Jun)
str(Jul)
str(Aug)
str(Sep)
str(Oct)
str(Nov)
str(Dec)

Noice! 

Looks like I just need to fix November's started_at / ended_at columns
since they aren't in a "datetime" format. 


In [ ]:
Nov <-  mutate(Nov, started_at = as.POSIXct(started_at, format="%m/%d/%Y %H:%M")
                   ,ended_at = as.POSIXct(ended_at, format = "%m/%d/%Y %H:%M"))

Now I'll verify structure again.

In [ ]:
str(Jan)
str(Feb)
str(Mar)
str(Apr)
str(May)
str(Jun)
str(Jul)
str(Aug)
str(Sep)
str(Oct)
str(Nov)
str(Dec)

Lookin good!

# **Now I'll combine the data into a new dataframe.**

In [ ]:
all_trips <- bind_rows(Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec)

**Check column names**

In [ ]:
colnames(all_trips)

**Take a sneak peek at the first 6 rows.**

In [ ]:
head(all_trips)

# Well... now what? I certainly can't make a meaningful analysis with my data like this, amirite?!


Break the datetime out:

* **date**
* **month**
* **day**
* **year**
* **day of week**

In [ ]:
all_trips$date <- as.Date(all_trips$started_at)
all_trips$month <- format(as.Date(all_trips$date), "%m")
all_trips$day <- format(as.Date(all_trips$date), "%d")
all_trips$year <- format(as.Date(all_trips$date), "%Y")
all_trips$day_of_week <- format(as.Date(all_trips$date), "%A")

In [ ]:
head(all_trips)

* Now, I want to see how long the members vs casuals have been riding around. 
* I'll create a calculated column to give me the amount of time (in seconds) between started_at and ended_at. 


In [ ]:
all_trips$ride_length <- difftime(all_trips$ended_at,all_trips$started_at)

In [ ]:
head(all_trips)

I'm going to change "ride_length" to numeric for future calculations.

In [ ]:
is.factor(all_trips$ride_length)
all_trips$ride_length <- as.numeric(as.character(all_trips$ride_length))
is.numeric(all_trips$ride_length)

One more check on our data :)

In [ ]:
str(all_trips)

**NOICE!**

# **Time to clean it up!**

* There are quite a few rows where my ride_length is negative. 
* I'll create new data frame called cleaned_trips with all rows omitted where ride_length isn't less than 0.

In [ ]:
cleaned_trips <- all_trips[!(all_trips$ride_length<0),]

In [ ]:
head(cleaned_trips)

Get the NULLs outta heeeere!

In [ ]:
cleaned_trips <- na.omit(cleaned_trips)

**Now, lemme get into this ride_length.**

In [ ]:
summary(cleaned_trips$ride_length)


Not bad, 23 days seems a bit extreme though... 

# Comparing members vs casuals.

In [ ]:
aggregate(cleaned_trips$ride_length ~ cleaned_trips$member_casual, FUN = mean)
aggregate(cleaned_trips$ride_length ~ cleaned_trips$member_casual, FUN = median)
aggregate(cleaned_trips$ride_length ~ cleaned_trips$member_casual, FUN = max)
aggregate(cleaned_trips$ride_length ~ cleaned_trips$member_casual, FUN = min)

Alright, looks like casuals beat out members when it comes to mean ride length!

**I wanna see their average by day now.**

Gonna go ahead and order days of the week here.

In [ ]:
cleaned_trips$day_of_week <- ordered(cleaned_trips$day_of_week, levels=c("Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"))

In [ ]:
aggregate(cleaned_trips$ride_length ~ cleaned_trips$member_casual + cleaned_trips$day_of_week, FUN = mean)


**Hmmmm... Maybe I should put a min/max limit on this data, I need to make this more practical.**

I don't care for trips that are less than 60 seconds or trips that are greater than 54,000 seconds (15 Hours) so let's get rid of those.

In [ ]:
cleaner_trips <- cleaned_trips[!(cleaned_trips$ride_length<60 | cleaned_trips$ride_length>54000),]

**Now, let's look at the cleaner_trips!**

In [ ]:
summary(cleaner_trips$ride_length)


In [ ]:
aggregate(cleaner_trips$ride_length ~ cleaner_trips$member_casual, FUN = mean)
aggregate(cleaner_trips$ride_length ~ cleaner_trips$member_casual, FUN = median)
aggregate(cleaner_trips$ride_length ~ cleaner_trips$member_casual, FUN = max)
aggregate(cleaner_trips$ride_length ~ cleaner_trips$member_casual, FUN = min)

I like it! Now, days of the week, going to go ahead and order the days before hand here as well.

In [ ]:
cleaner_trips$day_of_week <- ordered(cleaner_trips$day_of_week, levels=c("Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"))

In [ ]:
aggregate(cleaner_trips$ride_length ~ cleaner_trips$member_casual + cleaner_trips$day_of_week, FUN = mean)

**Not much of a change, but I feel much more comfortable with the data now without numerous extreme outliers.**

# **Let's analyze it!**

In [ ]:
nrow(cleaner_trips)

In [ ]:
table(cleaner_trips$member_casual)

I'll start with a basic pie chart showing our count of casuals vs members.

In [ ]:
colors <- c("#ff8c4f","#32efdc")

data <- cleaner_trips %>%
        group_by(member_casual) %>%
        summarize(count = n(),
        percentage = round(n()/nrow(cleaner_trips)*100,2), '%')

p <- plot_ly(data = data, labels = ~count, values = ~percentage, 
             type = 'pie', sort= FALSE,
            marker= list(colors=colors, line = list(color="black", width=1))) %>%
            layout(title="Casual vs Members")

In [ ]:
p

# After cleaning I'm left with 4,293,274 rides.

Of these, 2,562,562 (59%) are Members and 1,730,712 (41%) are Casual riders.

**We can definitely make this a little cleaner using plotly.**

In [ ]:
riders <- table(cleaner_trips$member_casual)
riders <- data.frame("category" = rownames(riders), riders)
data <- riders[,c('category','Freq')]
colors <- c('rgb(255,140,79)', 'rgb(50,239,220)', 'rgb(144,103,167)', 'rgb(171,104,87)', 'rgb(114,147,203)')


fig <- plot_ly(data, labels = ~category, values = ~Freq, type = 'pie',

        textposition = 'inside',

        textinfo = 'label+percent',

        insidetextfont = list(color = '#000000'),

        hoverinfo = 'value+text',

        text = ~paste(round(Freq/nrow(cleaner_trips)*100, 2), '%'),

        marker = list(colors = colors,

                      line = list(color = '#FFFFFF', width = 1)),

        showlegend = FALSE)

fig <- fig %>% layout(title = 'Casual Riders vs Members',

         xaxis = list(showgrid = FALSE, zeroline = FALSE, showticklabels = FALSE),

         yaxis = list(showgrid = FALSE, zeroline = FALSE, showticklabels = FALSE))

In [ ]:
fig

Let's take a look at the average ride length between stations of casuals vs members one more time.

In [ ]:
aggregate(cleaner_trips$ride_length ~ cleaner_trips$member_casual + cleaner_trips$day_of_week, FUN = mean)


**Let's throw in a bar chart!**

In [ ]:
cleaner_trips %>% 
  mutate(weekday = wday(started_at, label = TRUE)) %>% 
  group_by(member_casual, weekday) %>% 
  summarise(number_of_rides = n()
            ,average_duration = mean(ride_length)) %>% 
  arrange(member_casual, weekday)  %>% 
  ggplot(aes(x = weekday, y = average_duration, fill = member_casual)) +
  geom_col(position = "dodge")

Casuals definitely seem to be taking the bikes out for longer, these could be tourists. I'll need to look at station information to place more meaning to this. 

**Now I want to take a look at the amount of trips taken by casuals vs members.**

In [ ]:
cleaner_trips %>% 
  mutate(weekday = wday(started_at, label = TRUE)) %>% 
  group_by(member_casual, weekday) %>% 
  summarise(number_of_rides = n()
            ,average_duration = mean(ride_length)) %>%
  arrange(member_casual, weekday)


Ooof, kinda rough to look at, time for another chart!

In [ ]:
cleaner_trips %>% 
  mutate(weekday = wday(started_at, label = TRUE)) %>% 
  group_by(member_casual, weekday) %>% 
  summarise(number_of_rides = n()
            ,average_duration = mean(ride_length)) %>% 
  arrange(member_casual, weekday)  %>% 
  ggplot(aes(x = weekday, y = number_of_rides, fill = member_casual)) +
  geom_col(position = "dodge")

When it comes to the number of rides, members are definitely winning here. Monday - Friday has the heaviest traffic, members most likely cycle to and from work.

**Now, I want to create monthly data for analysis.**

In [ ]:
month_data <- cleaner_trips %>% 
  mutate(monthname = month(started_at, label = TRUE)) %>%
  group_by(member_casual, monthname) %>%
  summarise(number_of_rides = n()
            ,average_duration = mean(ride_length))

month_data

**Another one!**

In [ ]:
cleaner_trips %>% 
  mutate(month_data = month(started_at, label = TRUE)) %>% 
  group_by(member_casual, month) %>% 
  summarise(number_of_rides = n()
            ,average_duration = mean(ride_length)) %>% 
  arrange(member_casual, month)  %>% 
  ggplot(aes(x = month, y = number_of_rides, fill = member_casual)) +
  geom_col(position = "dodge")

There is clearly a preferred season for casuals.

# **Conclusion**

After looking over the data and analyzing trip duration paired with the number of daily and monthly rides for casuals and members, I have concluded the following:

* Casual riders are taking longer rides. *- Conversion to a member plan can be incentivized with a distance traveled:points system.*

* The number of casual rides increases significantly from May - September. *- Conversion is possible with a seasonal promotion.*

* Monday - Thursday casual rides are much lower than on Friday, Saturday, and Sunday. *- Conversion here can be accomplished with a "save the planet, cycle to work." ad campaign, combined with offering bonus points to casuals who make the switch.*


# Active Points System #

I suggest a points system as a versatile way of offering members unique rewards through an app.

Cyclistic could offer sign up bonus points and weekly points for completing challenges: 

>"Cycle to work 5 days this week for 1000 points!"

>"Log 5 miles of cycling this week for 500 points!"

* Cost savings for Cyclistic will come from less **overall** ad-spend, after the companion app is fully adopted, Cyclistic can rely more heavily on the app's notification "push" functionality. 
* The promotions users receive will organically convert to word-of-mouth advertising. 
* Once partnerships are established, points can be used with various industry leaders in fitness, clothing, health insurance, electricity providers, resturants, public transit, etc.

**Cyclistic can offer their members unique rewards and discounts they won't find anywhere else while also helping to reduce CO2 emissions.**  



# **Proposed Strategy**

The primary goal of this campaign is to convert casual users to members by offering the ability to gain points and exchange for unique rewards with cyclistic partners. A secondary goal is to encourage more casual users and members to cycle to and from work while positioning Cyclistic to make a positive change in the world. This campaign will also generate a lot of organic buzz about cyclistic by word of mouth. This campaign is heavy on the digital front, Cyclistic will gain exposure through popular entertainment services: youtube, tiktok, instagram, and google. Traditional advertising will also be incorporated into this campaign through billboards, bus ads and train ads. Cyclistic will build an internet presense around this movement through all forms of advertising mentioned as well as appear on top relevant google queries by utilizing a fine-tuned SEO management system that will create an organic online presence for Cyclistic.

**"Save the planet, cycle to work." - An Aggressive 6 Month Ad Campaign (May through October)**

During the 6 months this ad campaign is running, users will have the option to turn in points they have earned back to Cyclistic through the companion app. The points that are turned in will translate to a donation made to a charity of the users choice. Members who participate will receive a physical certificate for various milestones achieved during this event such as: 500 Points, 1,000 Points, 5,000 Points, 10,000 Points.

**User Donation Guidelines:**

* Cyclistic will make donations to various, relevant charities working to reduce emissions through advancements in technology and more efficient processes.
* For every 500 points turned in, Cyclistic will donate $5 to a partnered charity the Member chooses.
* Point boosts will be applied Monday through Friday during users' local "travel hours". 
* Travel hours will be 0400-0800 and 1700-2100.
* Points system only available to members.
 
**Action items:**

*  Create dedicated, content relevant, webpages. As well as backlinking, redirects, etc. tailored to adwords campaigns pushing this promotion out as effectively as possible for #1 on top 3 relevant queries.
*  Sponsor popular, relevant youtube channels with an ad spot in their videos for the next 6 months.
*  Hire Instagram and tiktok influencers to make posts
*  Billboard ads
*  Bus and Train ads


# **Data viz**

Full dataviz [here](https://public.tableau.com/app/profile/jonathan8167/viz/2022CyclisticData/Cyclistic2022) 

<script type="module" src="https://public.tableau.com/javascripts/api/tableau.embedding.3.latest.js"></script>

<!-- Use the width and height of the dashboard, with toolbar and tab adjustments as detailed above. -->
<div style="width:100%; height:100%; margin: 0 auto;">
  <tableau-viz id="tableauViz" src="https://public.tableau.com/views/2022CyclisticData/Cyclistic2022" tabs="none" toolbar="none" hide-tabs="true">
  </tableau-viz>
</div>